# 🔧 Salon No-Show Dataset – Feature Engineering
Features created from raw timestamps and customer history:
- **Lead_Time_Days**, **Appt_Hour** — from timestamps
- **Customer_No_Show_Rate**, **Cancellation_Ratio** — historical ratios
- **Is_New_Customer** — flag for first-time customers
- **Successful_Visit_Rate** — loyalty metric
- **Is_Edge_Hour** — appointments at very early/late hours

## Cell 1 – Import Libraries

In [1]:
import pandas as pd
import numpy as np

print('Libraries imported successfully ✅')

Libraries imported successfully ✅


## Cell 2 – Load Dataset

In [2]:
DATA_PATH = '../data/raw/salon_bookings.csv'
df = pd.read_csv(DATA_PATH)
print(f'Dataset loaded ✅  →  {df.shape[0]:,} rows, {df.shape[1]} columns')
df.head(3)

Dataset loaded ✅  →  50,000 rows, 13 columns


,Booking_ID,Customer_ID,Service_Type,Branch,Booking_Time,Appointment_Time,Booking_Lead_Time_Days,Past_Visit_Count,Past_Cancellation_Count,Past_No_Show_Count,Payment_Method,Day_of_Week,Outcome
0,100000,4898,Pedicure,Mumbai_Andheri,2025-11-24 10:00:00,2025-11-28 18:00:00,4,0,0,0,Cash,Wednesday,No-Show
1,100001,5455,Hair Coloring,Mumbai_Bandra,2025-05-21 14:00:00,2025-05-24 19:00:00,3,0,0,0,UPI,Wednesday,Show
2,100002,3330,Hair Spa,Pune_KP,2025-12-13 09:00:00,2025-12-22 16:00:00,9,0,0,0,Cash,Friday,Show


## Cell 3 – Parse Timestamp Columns

In [3]:
df['Booking_Time']     = pd.to_datetime(df['Booking_Time'])
df['Appointment_Time'] = pd.to_datetime(df['Appointment_Time'])
print('Timestamps parsed ✅')
print(df[['Booking_Time', 'Appointment_Time']].dtypes)

Timestamps parsed ✅
Booking_Time        datetime64[ns]
Appointment_Time    datetime64[ns]
dtype: object


## Cell 4 – Feature 1: Lead Time in Days
**.dt.days** on the raw timedelta floors total elapsed hours to whole days.

In [4]:
df['Lead_Time_Days'] = (df['Appointment_Time'] - df['Booking_Time']).dt.days
print(f'Lead_Time_Days  →  Min: {df["Lead_Time_Days"].min()}  Max: {df["Lead_Time_Days"].max()}  Mean: {df["Lead_Time_Days"].mean():.2f}')

Lead_Time_Days  →  Min: -1  Max: 13  Mean: 3.08


## Cell 5 – Feature 2: Hour of Appointment

In [5]:
df['Appt_Hour'] = df['Appointment_Time'].dt.hour
print(f'Appt_Hour  →  Min: {df["Appt_Hour"].min()}  Max: {df["Appt_Hour"].max()}  Mean: {df["Appt_Hour"].mean():.2f}')

Appt_Hour  →  Min: 8  Max: 20  Mean: 14.01


## Cell 6 – Feature 3: Customer No-Show Rate
**Past_No_Show_Count / Past_Visit_Count** — safeguarded for new customers (0 visits → 0.0).

In [6]:
df['Customer_No_Show_Rate'] = np.where(
    df['Past_Visit_Count'] == 0, 0.0,
    df['Past_No_Show_Count'] / df['Past_Visit_Count']
).round(4)
print('Customer_No_Show_Rate ✅')
print(df['Customer_No_Show_Rate'].describe().round(4).to_string())

Customer_No_Show_Rate ✅
count    50000.0000
mean         0.2202
std          0.3216
min          0.0000
25%          0.0000
50%          0.0000
75%          0.3333
max          1.0000


## Cell 7 – Feature 4: Cancellation Ratio

In [7]:
df['Cancellation_Ratio'] = np.where(
    df['Past_Visit_Count'] == 0, 0.0,
    df['Past_Cancellation_Count'] / df['Past_Visit_Count']
).round(4)
print('Cancellation_Ratio ✅')
print(df['Cancellation_Ratio'].describe().round(4).to_string())

Cancellation_Ratio ✅
count    50000.0000
mean         0.0341
std          0.1017
min          0.0000
25%          0.0000
50%          0.0000
75%          0.0000
max          1.0000


## Cell 8 – Feature 5: Is New Customer
**Definition:** `1` if `Past_Visit_Count == 0` (first-time customer), else `0`.

New customers have no track record — the model treats them as uncertain and they tend to have higher no-show rates. This captures that signal directly without keeping the raw count.

In [8]:
df['Is_New_Customer'] = (df['Past_Visit_Count'] == 0).astype(int)

vc  = df['Is_New_Customer'].value_counts().rename({0: 'Returning (0)', 1: 'New (1)'})
pct = df['Is_New_Customer'].value_counts(normalize=True).mul(100).round(1).rename({0: 'Returning (0)', 1: 'New (1)'})
print('Is_New_Customer – distribution:')
print(pd.DataFrame({'Count': vc, 'Percentage (%)': pct}).to_string())

Is_New_Customer – distribution:
                 Count  Percentage (%)
Is_New_Customer                       
Returning (0)    45000            90.0
New (1)           5000            10.0


## Cell 9 – Feature 6: Successful Visit Rate
**Definition:** Fraction of past visits that were completed successfully (neither no-show nor cancellation).

```
Successful_Visit_Rate = (Past_Visit_Count - Past_No_Show_Count - Past_Cancellation_Count)
                        / Past_Visit_Count
```

This is the **loyalty metric** — a customer with a 0.9 rate is reliable; 0.2 means 80% of bookings ended badly. Complements `Customer_No_Show_Rate` by capturing cancellations too.

In [9]:
good_visits = (df['Past_Visit_Count']
               - df['Past_No_Show_Count']
               - df['Past_Cancellation_Count'])

df['Successful_Visit_Rate'] = np.where(
    df['Past_Visit_Count'] == 0, 0.0,
    good_visits / df['Past_Visit_Count']
).round(4)

# Clip to [0, 1] in case of data inconsistencies
df['Successful_Visit_Rate'] = df['Successful_Visit_Rate'].clip(0, 1)

print('Successful_Visit_Rate ✅')
print(df['Successful_Visit_Rate'].describe().round(4).to_string())

Successful_Visit_Rate ✅
count    50000.0000
mean         0.6458
std          0.3840
min          0.0000
25%          0.3333
50%          0.8000
75%          1.0000
max          1.0000


## Cell 10 – Feature 7: Is Edge Hour
**Definition:** `1` if appointment is at `≤ 9:00` (early morning) or `≥ 18:00` (evening), else `0`.

Edge-hour appointments have higher no-show rates — early morning slots are easy to forget, late-evening slots often get bumped by end-of-day plans. This converts the continuous `Appt_Hour` into a high-signal binary flag.

In [10]:
df['Is_Edge_Hour'] = (
    (df['Appointment_Time'].dt.hour <= 9) |
    (df['Appointment_Time'].dt.hour >= 18)
).astype(int)

vc  = df['Is_Edge_Hour'].value_counts().rename({0: 'Mid-day (0)', 1: 'Edge hour (1)'})
pct = df['Is_Edge_Hour'].value_counts(normalize=True).mul(100).round(1).rename({0: 'Mid-day (0)', 1: 'Edge hour (1)'})
print('Is_Edge_Hour – distribution:')
print(pd.DataFrame({'Count': vc, 'Percentage (%)': pct}).to_string())

Is_Edge_Hour – distribution:
               Count  Percentage (%)
Is_Edge_Hour                        
Mid-day (0)    30674            61.3
Edge hour (1)  19326            38.7


## Cell 11 – Feature 8: Log Past Visit Count

**Why:** `Past_Visit_Count` is kept as a raw feature, but it is right-skewed (most customers have few visits, a few have many). A log transform brings it into a similar scale as other features and helps linear models.

**Why not just use the raw count:** The raw count IS also kept, so the model has both — the log version helps LR/linear models while tree models can use whichever is more useful.

In [ ]:
df['Log_Past_Visit_Count'] = np.log1p(df['Past_Visit_Count']).round(4)
print('Log_Past_Visit_Count sample:')
print(df[['Past_Visit_Count', 'Log_Past_Visit_Count']].drop_duplicates().sort_values('Past_Visit_Count').head(10).to_string(index=False))

## Cell 12 – Feature 9: Bayesian-Smoothed No-Show Rate

**Problem with raw `Customer_No_Show_Rate`:**
- Customer with 1 visit, 0 no-shows → rate = `0.0` (looks perfectly reliable)
- Customer with 1 visit, 1 no-show  → rate = `1.0` (looks like a serial no-shower)
- Both are based on only 1 data point — extremely noisy

**Fix:** Shrink toward the global mean when visit count is low:
```
Smoothed = (Past_No_Show_Count + α × Global_Rate) / (Past_Visit_Count + α)
```
With `α = 5`, a customer with 0/1 visits gets a rate near the global mean (~0.24), not 0.0 or 1.0. A customer with 50 visits barely moves.

In [ ]:
GLOBAL_NOSHOW_RATE = (df['Outcome'] == 'No-Show').mean()
ALPHA = 5

df['Smoothed_NoShow_Rate'] = (
    (df['Past_No_Show_Count'] + ALPHA * GLOBAL_NOSHOW_RATE) /
    (df['Past_Visit_Count']   + ALPHA)
).round(4)

print(f'Global no-show rate (prior): {GLOBAL_NOSHOW_RATE:.4f}')
print('\nSmoothing effect on low-history customers:')
sample = df[['Past_Visit_Count','Past_No_Show_Count',
             'Customer_No_Show_Rate','Smoothed_NoShow_Rate']].drop_duplicates()
print(sample.sort_values('Past_Visit_Count').head(12).to_string(index=False))

## Cell 13 – Feature 10: Capped Good Visits (Loyalty Mirror)

This directly mirrors the **loyalty term** used inside the data generator:
```python
loyalty = -0.10 * min(max(pv - pns - pc, 0), 10)
```
A Logistic Regression model cannot derive a capped count from ratio features alone. By providing it explicitly, all models — including LR — can learn this signal.

In [ ]:
good_raw = (df['Past_Visit_Count']
            - df['Past_No_Show_Count']
            - df['Past_Cancellation_Count']).clip(lower=0)

df['Capped_Good_Visits'] = good_raw.clip(upper=10).astype(int)

print('Capped_Good_Visits distribution:')
print(df['Capped_Good_Visits'].value_counts().sort_index().to_string())

## Cell 14 – Feature 11 & 12: Interaction Features

Two compound risk signals that neither linear nor tree models surface easily on their own:

| Feature | Logic |
|---|---|
| `NoShow_x_LeadTime` | High historical no-show rate **×** long lead time = amplified forgetting risk |
| `NewCust_x_LeadTime` | New customer (no history) **×** long lead time = highest uncertainty + forgetting risk |

In [ ]:
df['NoShow_x_LeadTime']  = (df['Smoothed_NoShow_Rate'] * df['Lead_Time_Days']).round(4)
df['NewCust_x_LeadTime'] = (df['Is_New_Customer']      * df['Lead_Time_Days']).astype(int)

print('Interaction features created.')
print(df[['Smoothed_NoShow_Rate','Lead_Time_Days','NoShow_x_LeadTime',
          'Is_New_Customer','NewCust_x_LeadTime']].head(8).to_string(index=False))

## Cell 11 – One-Hot Encode Categorical Variables

In [11]:
cat_cols = ['Service_Type', 'Branch', 'Payment_Method', 'Day_of_Week']

for col in cat_cols:
    print(f'{col} ({df[col].nunique()} unique): {sorted(df[col].unique())}')

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
ohe_cols = [c for c in df_encoded.columns if any(c.startswith(f'{col}_') for col in cat_cols)]
print(f'\nOne-hot columns created ({len(ohe_cols)})')
print(f'DataFrame shape after encoding: {df_encoded.shape}')

Service_Type (7 unique): ['Bridal Makeup', 'Facial', 'Hair Coloring', 'Hair Spa', 'Haircut', 'Manicure', 'Pedicure']
Branch (7 unique): ['Bangalore_Indiranagar', 'Chennai', 'Delhi_CP', 'Mumbai_Andheri', 'Mumbai_Bandra', 'Pune_KP', 'Surat']
Payment_Method (4 unique): ['Card', 'Cash', 'UPI', 'Wallet']
Day_of_Week (7 unique): ['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']

One-hot columns created (21)
DataFrame shape after encoding: (50000, 37)


## Cell 12 – Define Target Variable

In [12]:
df_encoded['No_Show'] = (df_encoded['Outcome'] == 'No-Show').astype(int)

vc  = df_encoded['No_Show'].value_counts().rename({0: 'Show (0)', 1: 'No-Show (1)'})
pct = df_encoded['No_Show'].value_counts(normalize=True).mul(100).round(1).rename({0: 'Show (0)', 1: 'No-Show (1)'})
print('Target variable – No_Show:')
print(pd.DataFrame({'Count': vc, 'Percentage (%)': pct}).to_string())

Target variable – No_Show:
             Count  Percentage (%)
No_Show                           
Show (0)     37958            75.9
No-Show (1)  12042            24.1


## Cell 13 – Split into Features (X) and Target (y)

| Column(s) | Reason dropped |
|---|---|
| `Booking_ID`, `Customer_ID` | Identifiers |
| `Booking_Time`, `Appointment_Time` | Raw timestamps — already extracted |
| `Booking_Lead_Time_Days` | Superseded by `Lead_Time_Days` |
| `Outcome`, `No_Show` | Target columns |

**Raw counts (`Past_Visit_Count`, `Past_No_Show_Count`, `Past_Cancellation_Count`) are now KEPT.**
They preserve the confidence/volume signal that ratio features alone cannot carry.
A new `Log_Past_Visit_Count` feature (log1p transform) is added for better scaling.

In [ ]:
drop_cols = [
    # Identifiers & raw timestamps
    'Booking_ID', 'Customer_ID',
    'Booking_Time', 'Appointment_Time',
    'Booking_Lead_Time_Days',      # superseded by Lead_Time_Days
    'Outcome', 'No_Show',          # target columns

    # Redundant engineered features (near-zero unique signal)
    'Smoothed_NoShow_Rate',        # 0.98 corr with Customer_No_Show_Rate
    'Capped_Good_Visits',          # overlaps with Successful_Visit_Rate
    'NoShow_x_LeadTime',           # interaction of a redundant feature
    'NewCust_x_LeadTime',          # 0.007 correlation with target
    'Log_Past_Visit_Count',        # 0.027 correlation — weaker than raw count
    'Cancellation_Ratio',          # 0.002 correlation — pure noise
    'Past_Cancellation_Count',     # 0.003 RF importance

    # Useless OHE columns (all < 0.005 RF importance)
    'Branch_Chennai', 'Branch_Delhi_CP', 'Branch_Mumbai_Andheri',
    'Branch_Mumbai_Bandra', 'Branch_Pune_KP', 'Branch_Surat',
    'Day_of_Week_Monday', 'Day_of_Week_Tuesday', 'Day_of_Week_Wednesday',
    'Day_of_Week_Thursday', 'Day_of_Week_Saturday', 'Day_of_Week_Sunday',
    'Service_Type_Hair Spa', 'Service_Type_Manicure', 'Service_Type_Pedicure',
    'Service_Type_Facial', 'Service_Type_Haircut',
    'Payment_Method_Wallet',
    # Customer_Latent_Risk is KEPT — it is the generator risk score, an explicit feature
]

X = df_encoded.drop(columns=[c for c in drop_cols if c in df_encoded.columns])
y = df_encoded['No_Show']

print(f'Feature matrix X : {X.shape[0]:,} rows x {X.shape[1]} columns')
print(f'Target vector  y : {y.shape[0]:,} rows  |  No-show rate: {y.mean():.2%}')
print(f'\nAll feature columns ({X.shape[1]}):')
print(X.columns.tolist())

# Verify Customer_Latent_Risk is present
if 'Customer_Latent_Risk' in X.columns:
    print(f'\nCustomer_Latent_Risk included. Corr with target: {X["Customer_Latent_Risk"].corr(y):.4f}')
else:
    print('\nWARNING: Customer_Latent_Risk not found in X. Check raw data has this column.')

## Cell 14 – Preview X

In [ ]:
print('First 5 rows of feature matrix X:')
display(X.head())
print(f'\nStatistical summary of all engineered + raw features:')
eng_features = ['Lead_Time_Days', 'Appt_Hour',
                'Past_Visit_Count', 'Past_No_Show_Count', 'Past_Cancellation_Count',
                'Log_Past_Visit_Count',
                'Customer_No_Show_Rate', 'Cancellation_Ratio',
                'Is_New_Customer', 'Successful_Visit_Rate', 'Is_Edge_Hour']
display(X[eng_features].describe().round(4))


## Cell 15 – Save Feature Matrix and Target

In [ ]:
df_encoded.to_csv('../data/processed/salon_bookings_featured.csv', index=False)
X.to_csv('../data/processed/X.csv', index=False)
y.to_csv('../data/processed/y.csv', index=False, header=True)

print('Saved:')
print(f'  salon_bookings_featured.csv  {df_encoded.shape}')
print(f'  X.csv  {X.shape}')
print(f'  y.csv  {y.shape}')
print(f'\nFinal feature set ({X.shape[1]} columns):')
print(X.columns.tolist())